# XFeat-SuperPoint Hybrid — MegaDepth Raw Training (Colab)

This notebook trains the model using `train.py --mode megadepth_raw`.

## Expected raw layout
```
<RAW_ROOT>/
  0001/dense0/imgs/*.jpg
  0001/dense0/depths/*.h5   (optional in current raw mode)
  0002/dense0/imgs/*.jpg
  ...
```

> Note: current `megadepth_raw` pairing uses images (`dense0/imgs`) and estimates geometry from image pairs.
> Depth `.h5` files are not consumed in this raw loader path.


In [ ]:
#@title 1) Mount Drive
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
#@title 2) Paths and knobs
from pathlib import Path

# --- CHANGE THESE ---
DRIVE_PROJECT_ROOT = '/content/drive/MyDrive/hybrid_feature_matching'
RAW_ROOT_DRIVE    = f'{DRIVE_PROJECT_ROOT}/data/megadepth_raw'  # folder containing scene dirs (0001, 0002, ...)
RAW_TAR_PATH      = ''  # optional: '/content/drive/MyDrive/.../megadepth_raw.tar'

REPO_DIR          = '/content/XFeat-SuperPoint-Hybrid-Model'
DRIVE_CKPT_DIR    = f'{DRIVE_PROJECT_ROOT}/checkpoints/megadepth_raw'

# training hyperparams
BATCH_SIZE        = 4
MAX_EPOCHS        = 50
LR                = 1e-4
NUM_WORKERS       = 2
IMAGE_H, IMAGE_W  = 480, 640   # must be divisible by 8
VAL_SPLIT_RATIO   = 0.2
MAX_PAIRS_PER_SCENE = 200
MIN_OVERLAP       = 0.15
MAX_OVERLAP       = 0.95

# optional resume checkpoint (leave empty to auto-pick best/last)
RESUME_CKPT       = ''

Path(DRIVE_CKPT_DIR).mkdir(parents=True, exist_ok=True)
print('REPO_DIR      :', REPO_DIR)
print('RAW_ROOT_DRIVE:', RAW_ROOT_DRIVE)
print('DRIVE_CKPT_DIR:', DRIVE_CKPT_DIR)


In [ ]:
#@title 3) Setup repo + dependencies
import os, subprocess
from pathlib import Path

if not Path(REPO_DIR).exists():
    subprocess.check_call(['git', 'clone', 'https://github.com/MalharRane/XFeat-SuperPoint-Hybrid-Model.git', REPO_DIR])

subprocess.check_call(['python', '-m', 'pip', 'install', '-q', '--upgrade', 'pip'])
subprocess.check_call(['python', '-m', 'pip', 'install', '-q', '-r', f'{REPO_DIR}/requirements.txt'])

print('Setup complete')


In [ ]:
#@title 4) Optional: extract TAR to /content if RAW_TAR_PATH is set
import os, tarfile
from pathlib import Path

if RAW_TAR_PATH:
    extract_root = Path('/content/megadepth_raw')
    flag = extract_root / '.extracted'
    if flag.exists():
        print('Already extracted:', extract_root)
    else:
        extract_root.mkdir(parents=True, exist_ok=True)
        print('Extracting', RAW_TAR_PATH, '->', extract_root)
        with tarfile.open(RAW_TAR_PATH, 'r') as tf:
            tf.extractall(extract_root)
        flag.write_text('ok')
    RAW_ROOT = str(extract_root)
else:
    RAW_ROOT = RAW_ROOT_DRIVE

print('RAW_ROOT:', RAW_ROOT)


In [ ]:
#@title 5) Validate raw dataset structure
from pathlib import Path

raw = Path(RAW_ROOT)
assert raw.exists(), f'RAW_ROOT does not exist: {raw}'

scene_dirs = sorted([p for p in raw.iterdir() if p.is_dir()])
img_dirs = list(raw.glob('**/dense0/imgs'))
depth_dirs = list(raw.glob('**/dense0/depths'))

num_imgs = 0
for d in img_dirs[:200]:
    num_imgs += len(list(d.glob('*.jpg'))) + len(list(d.glob('*.jpeg'))) + len(list(d.glob('*.png')))

num_h5 = 0
for d in depth_dirs[:200]:
    num_h5 += len(list(d.glob('*.h5')))

print('scene dirs :', len(scene_dirs))
print('img dirs   :', len(img_dirs))
print('depth dirs :', len(depth_dirs))
print('sample imgs count (partial scan):', num_imgs)
print('sample h5 count   (partial scan):', num_h5)

if len(img_dirs) == 0:
    raise RuntimeError('No dense0/imgs folders found. Check RAW_ROOT layout.')


In [ ]:
#@title 6) Pick resume checkpoint (optional auto-pick)
from pathlib import Path

resume = RESUME_CKPT.strip()
if not resume:
    ckpt_dir = Path(DRIVE_CKPT_DIR)
    best = ckpt_dir / 'best.pth'
    if best.exists():
        resume = str(best)
    else:
        epochs = sorted(ckpt_dir.glob('epoch_*.pth'))
        if epochs:
            resume = str(epochs[-1])

print('resume checkpoint:', resume if resume else '(none)')


In [ ]:
#@title 7) Train on MegaDepth Raw
import os, subprocess

cmd = [
    'python', f'{REPO_DIR}/train.py',
    '--mode', 'megadepth_raw',
    '--data_root', RAW_ROOT,
    '--checkpoint_dir', DRIVE_CKPT_DIR,
    '--batch_size', str(BATCH_SIZE),
    '--max_epochs', str(MAX_EPOCHS),
    '--lr', str(LR),
    '--num_workers', str(NUM_WORKERS),
    '--image_size', str(IMAGE_H), str(IMAGE_W),
    '--megadepth_val_split_ratio', str(VAL_SPLIT_RATIO),
    '--max_pairs_per_scene', str(MAX_PAIRS_PER_SCENE),
    '--min_overlap', str(MIN_OVERLAP),
    '--max_overlap', str(MAX_OVERLAP),
]

if resume:
    cmd += ['--resume', resume]

print('Running:', ' '.join(cmd))
subprocess.check_call(cmd)


In [ ]:
#@title 8) (Optional) Two-stage training: synthetic -> MegaDepth Raw
# Fill these before running this cell.
SYNTH_ROOT = ''  # e.g. '/content/drive/MyDrive/.../coco/train2017'
STAGE1_EPOCHS = 30
STAGE2_EPOCHS = 50

if SYNTH_ROOT:
    import subprocess
    cmd = [
        'python', f'{REPO_DIR}/train.py',
        '--two_stage',
        '--synthetic_data_root', SYNTH_ROOT,
        '--megadepth_data_root', RAW_ROOT,
        '--stage2_mode', 'megadepth_raw',
        '--stage1_epochs', str(STAGE1_EPOCHS),
        '--stage2_epochs', str(STAGE2_EPOCHS),
        '--checkpoint_dir', DRIVE_CKPT_DIR,
        '--batch_size', str(BATCH_SIZE),
        '--num_workers', str(NUM_WORKERS),
    ]
    print('Running:', ' '.join(cmd))
    subprocess.check_call(cmd)
else:
    print('Set SYNTH_ROOT first if you want two-stage training.')


In [ ]:
#@title 9) List saved checkpoints
from pathlib import Path

ckpts = sorted(Path(DRIVE_CKPT_DIR).glob('*.pth'))
print('Found', len(ckpts), 'checkpoint files')
for p in ckpts[-20:]:
    print(p.name)
